In [1]:
import pandas as pd
import torch
from tqdm import tqdm
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
from sklearn.metrics import f1_score, cohen_kappa_score, recall_score, accuracy_score, classification_report
from scipy.stats import spearmanr
from torch.utils.data import WeightedRandomSampler


# The following code will only execute
# successfully when compression is complete

import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohabenloughmari/miccai-task2-full")

print("Path to dataset files:", path)

import warnings
warnings.filterwarnings("ignore")

base_path = path
path = os.path.join(base_path, 'Task_2')

df_train = pd.read_csv(os.path.join(path, "df_task2_train.csv"))
df_val = pd.read_csv(os.path.join(path, "df_task2_val.csv"))
df_test = pd.read_csv(os.path.join(path, "df_task2_test.csv"))

device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')


Using Colab cache for faster access to the 'miccai-task2-full' dataset.
Path to dataset files: /kaggle/input/miccai-task2-full


In [2]:

labels = df_train['label'].values.astype(int)
class_counts = np.bincount(labels)
sample_weights = 1.0 / class_counts[labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

n_labels = df_train['label'].nunique()

tabular_cols = ['case', 'label', 'LOCALIZER', 'split_type', 'image']
tab_train = df_train.drop(columns=tabular_cols)
tab_val = df_val.drop(columns=tabular_cols)
tab_test = df_test.drop(columns=tabular_cols)

cat_cols = tab_train.select_dtypes(include='object').columns.tolist()
num_cols = tab_train.select_dtypes(exclude='object').columns.tolist()

print(f"Categorical columns: {cat_cols}")
print(f"Numerical columns: {num_cols}")

tab_train_encoded = pd.get_dummies(tab_train, columns=cat_cols, dtype=np.float32)
tab_val_encoded = pd.get_dummies(tab_val, columns=cat_cols, dtype=np.float32)
tab_test_encoded = pd.get_dummies(tab_test, columns=cat_cols, dtype=np.float32)

tab_val_encoded = tab_val_encoded.reindex(columns=tab_train_encoded.columns, fill_value=0)
tab_test_encoded = tab_test_encoded.reindex(columns=tab_train_encoded.columns, fill_value=0)



Categorical columns: ['side_eye', 'sex']
Numerical columns: ['BScan', 'age', 'num_current_visit']


In [3]:

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [4]:

class CustomImageDataset(Dataset):
    def __init__(self, dataframe, root_dir, tab_data, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.tab_data = torch.tensor(tab_data.values, dtype=torch.float32)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['image'])
        localiser_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['LOCALIZER'])

        image = Image.open(img_name).convert('RGB')
        localiser = Image.open(localiser_name).convert('RGB')

        if self.transform:
            image = self.transform(image)
            localiser = self.transform(localiser)

        label = self.dataframe.iloc[idx]['label']
        tab = self.tab_data[idx]
        return image, localiser, tab, label

In [5]:
train_ds = CustomImageDataset(df_train, os.path.join(path, 'train'), tab_train_encoded, transform=train_transform)  
val_ds = CustomImageDataset(df_val, os.path.join(path, 'val'), tab_val_encoded, transform=test_transform)  
test_ds = CustomImageDataset(df_test, os.path.join(path, 'test'), tab_test_encoded, transform=test_transform)  
  
labels = df_train['label'].values.astype(int)  
class_counts = np.bincount(labels)  
sample_weights = 1.0 / class_counts[labels]  
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))  
  
train_loader = DataLoader(train_ds, batch_size=64, sampler=sampler)  
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)  
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

In [6]:
from tqdm import tqdm
import torch  
import torch.nn as nn  
import numpy as np  
from torchvision import models  
from sklearn.linear_model import LogisticRegression  
from sklearn.preprocessing import StandardScaler  
  
  
class FeatureExtractor(nn.Module):  
    """Frozen EfficientNetV2-S feature extractor."""  
    def __init__(self):  
        super().__init__()  
        weights = models.EfficientNet_V2_S_Weights.DEFAULT  
        self.backbone = models.efficientnet_v2_s(weights=weights)  
        self.backbone.classifier = nn.Identity()  
        for p in self.backbone.parameters():  
            p.requires_grad = False  
  
    @torch.no_grad()  
    def forward(self, x):  
        return self.backbone(x)


def extract_features(dataloader, device):
    """Extract and concatenate image + tabular features from a dataloader."""
    extractor = FeatureExtractor().to(device).eval()
    all_feats = []
    all_labels = []
    for batch in tqdm(dataloader, desc="Extracting features"):
        images, localisers, tab, labels = batch
        images = images.to(device)
        img_feats = extractor(images).cpu().numpy()
        tab_np = tab.numpy() if isinstance(tab, torch.Tensor) else np.array(tab)
        combined = np.concatenate([img_feats, tab_np], axis=1)
        all_feats.append(combined)
        all_labels.append(labels.numpy() if isinstance(labels, torch.Tensor) else labels)
    feats = np.concatenate(all_feats)
    labels = np.concatenate(all_labels)
    return feats, labels


def build_logreg_pipeline(train_loader, val_loader, device):
    """Full pipeline: extract (image + tab) → StandardScaler → LogisticRegression."""
    print("Extracting train features...")
    X_train, y_train = extract_features(train_loader, device)
    print("Extracting val features...")
    X_val, y_val = extract_features(val_loader, device)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    class_weights_dict = {i: 1.0 / c for i, c in enumerate(class_counts)}
    clf = LogisticRegression(
        max_iter=1000,
        class_weight=class_weights_dict,
        multi_class='multinomial',
        solver='lbfgs',
    )
    clf.fit(X_train, y_train)

    train_acc = clf.score(X_train, y_train)
    val_acc = clf.score(X_val, y_val)
    print(f"Train accuracy: {train_acc:.4f}")
    print(f"Val accuracy:   {val_acc:.4f}")

    return scaler, clf
  
  
# --- Usage ---  
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  
scaler, clf = build_logreg_pipeline(train_loader, test_loader, device)

Extracting train features...
Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 189MB/s]
Extracting features: 100%|██████████| 127/127 [08:52<00:00,  4.20s/it]


Extracting val features...


Extracting features: 100%|██████████| 90/90 [05:11<00:00,  3.46s/it]


Train accuracy: 0.5562
Val accuracy:   0.0729


In [7]:
from tqdm import tqdm
import torch  
import torch.nn as nn  
import numpy as np  
from torchvision import models  
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier


class FeatureExtractor(nn.Module):  
    """Frozen EfficientNetV2-S feature extractor."""  
    def __init__(self):  
        super().__init__()  
        weights = models.EfficientNet_V2_S_Weights.DEFAULT  
        self.backbone = models.efficientnet_v2_s(weights=weights)  
        self.backbone.classifier = nn.Identity()  
        for p in self.backbone.parameters():  
            p.requires_grad = False  
  
    @torch.no_grad()  
    def forward(self, x):  
        return self.backbone(x)


def extract_features(dataloader, device):
    """Extract and concatenate image + tabular features from a dataloader."""
    extractor = FeatureExtractor().to(device).eval()
    all_feats = []
    all_labels = []
    for batch in tqdm(dataloader, desc="Extracting features"):
        images, localisers, tab, labels = batch
        images = images.to(device)
        img_feats = extractor(images).cpu().numpy()
        tab_np = tab.numpy() if isinstance(tab, torch.Tensor) else np.array(tab)
        combined = np.concatenate([img_feats, tab_np], axis=1)
        all_feats.append(combined)
        all_labels.append(labels.numpy() if isinstance(labels, torch.Tensor) else labels)
    feats = np.concatenate(all_feats)
    labels = np.concatenate(all_labels)
    return feats, labels


def build_xgb_pipeline(train_loader, val_loader, device):
    """Full pipeline: extract (image + tab) → StandardScaler → XGBClassifier."""
    print("Extracting train features...")
    X_train, y_train = extract_features(train_loader, device)
    print("Extracting val features...")
    X_val, y_val = extract_features(val_loader, device)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    # Compute sample weights from class counts
    sample_weights = np.array([1.0 / class_counts[int(y)] for y in y_train])

    clf = XGBClassifier(
        n_estimators=300,
        learning_rate=0.1,
        max_depth=6,
        use_label_encoder=False,
        eval_metric="mlogloss",
        device="cuda" if torch.cuda.is_available() else "cpu",
    )
    clf.fit(
        X_train, y_train,
        sample_weight=sample_weights,
        eval_set=[(X_val, y_val)],
        verbose=50,
    )

    train_acc = (clf.predict(X_train) == y_train).mean()
    val_acc = (clf.predict(X_val) == y_val).mean()
    print(f"Train accuracy: {train_acc:.4f}")
    print(f"Val accuracy:   {val_acc:.4f}")
    return scaler, clf


# --- Usage ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler, clf = build_xgb_pipeline(train_loader, test_loader, device)

Extracting train features...


Extracting features: 100%|██████████| 127/127 [08:56<00:00,  4.23s/it]


Extracting val features...


Extracting features: 100%|██████████| 90/90 [05:04<00:00,  3.38s/it]


[0]	validation_0-mlogloss:2.70008
[50]	validation_0-mlogloss:2.70205
[100]	validation_0-mlogloss:2.72372
[150]	validation_0-mlogloss:2.73918
[200]	validation_0-mlogloss:2.75056
[250]	validation_0-mlogloss:2.75633
[299]	validation_0-mlogloss:2.75838
Train accuracy: 0.4598
Val accuracy:   0.0671


In [ ]:
from tqdm import tqdm

import torch  
import torch.nn as nn  
import numpy as np  
from torchvision import models  
from sklearn.decomposition import PCA  
from sklearn.linear_model import LogisticRegression  
from sklearn.preprocessing import StandardScaler  
  
  
class FeatureExtractor(nn.Module):  
    """Frozen EfficientNetV2-S feature extractor."""  
    def __init__(self):  
        super().__init__()  
        weights = models.EfficientNet_V2_S_Weights.DEFAULT  
        self.backbone = models.efficientnet_v2_s(weights=weights)  
        self.backbone.classifier = nn.Identity()  
        for p in self.backbone.parameters():  
            p.requires_grad = False  
  
    @torch.no_grad()  
    def forward(self, x):  
        return self.backbone(x)
def extract_features(dataloader, device):
    """Extract features from a dataloader using frozen EfficientNetV2."""
    extractor = FeatureExtractor().to(device).eval()

    all_img_feats = []
    all_labels = []

    for batch in tqdm(dataloader, desc="Extracting features"):
        images, localisers, tab, labels = batch
        images = images.to(device)

        img_feats = extractor(images).cpu().numpy()

        all_img_feats.append(img_feats)
        all_labels.append(labels.numpy() if isinstance(labels, torch.Tensor) else labels)

    img_feats = np.concatenate(all_img_feats)
    labels = np.concatenate(all_labels)

    return img_feats, labels


def build_pca_logreg_pipeline(train_loader, val_loader, device, n_components=10):
    """Full pipeline: extract → PCA(3) → LogisticRegression."""

    print("Extracting train features...")
    X_train, y_train = extract_features(train_loader, device)
    print("Extracting val features...")
    X_val, y_val = extract_features(val_loader, device)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)

    pca = PCA(n_components=n_components)
    X_train_pca = pca.fit_transform(X_train)
    X_val_pca = pca.transform(X_val)
    print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
    print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.4f}")

    class_weights_dict = {i: 1.0 / c for i, c in enumerate(class_counts)}
    clf = LogisticRegression(
        max_iter=1000,
        class_weight=class_weights_dict,
        multi_class='multinomial',
        solver='lbfgs',
    )
    clf.fit(X_train_pca, y_train)

    train_acc = clf.score(X_train_pca, y_train)
    val_acc = clf.score(X_val_pca, y_val)
    print(f"Train accuracy: {train_acc:.4f}")
    print(f"Val accuracy:   {val_acc:.4f}")

    return scaler, pca, clf
  
# --- Usage ---  
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  
scaler, pca, clf = build_pca_logreg_pipeline(train_loader, test_loader, device, n_components=10)

In [ ]:
!pip install open_clip_torch

In [ ]:
#import torch
#import numpy as np
#from tqdm import tqdm
#from sklearn.decomposition import PCA
#from sklearn.linear_model import LogisticRegression
#from sklearn.preprocessing import StandardScaler
#from open_clip import create_model_from_pretrained, get_tokenizer
#
#
#class FeatureExtractor:
#    """Frozen BiomedCLIP feature extractor for OCT images."""
#    def __init__(self, device):
#        self.device = device
#        self.model, self.preprocess = create_model_from_pretrained(
#            'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
#        )
#        self.model = self.model.to(device).eval()
#        for p in self.model.parameters():
#            p.requires_grad = False
#
#    @torch.no_grad()
#    def __call__(self, images):
#        images = images.to(self.device)
#        features = self.model.encode_image(images)
#        return features.cpu().numpy()
#
#
#def extract_features(dataloader, extractor):
#    """Extract features from a dataloader using frozen BiomedCLIP."""
#    all_img_feats = []
#    all_labels = []
#
#    for batch in tqdm(dataloader, desc="Extracting features"):
#        images, localisers, tab, labels = batch
#
#        img_feats = extractor(images)
#
#        all_img_feats.append(img_feats)
#        all_labels.append(labels.numpy() if isinstance(labels, torch.Tensor) else labels)
#
#    img_feats = np.concatenate(all_img_feats)
#    labels = np.concatenate(all_labels)
#
#    return img_feats, labels
#
#
#def build_pca_logreg_pipeline(train_loader, val_loader, device, n_components=3):
#    """Full pipeline: BiomedCLIP extract → PCA(3) → LogisticRegression."""
#
#    extractor = FeatureExtractor(device)
#
#    print("Extracting train features...")
#    X_train, y_train = extract_features(train_loader, extractor)
#    print("Extracting val features...")
#    X_val, y_val = extract_features(val_loader, extractor)
#
#    scaler = StandardScaler()
#    X_train = scaler.fit_transform(X_train)
#    X_val = scaler.transform(X_val)
#
#    pca = PCA(n_components=n_components)
#    X_train_pca = pca.fit_transform(X_train)
#    X_val_pca = pca.transform(X_val)
#    print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
#    print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.4f}")
#
#    class_weights_dict = {i: 1.0 / c for i, c in enumerate(class_counts)}
#    clf = LogisticRegression(
#        max_iter=1000,
#        class_weight=class_weights_dict,
#        multi_class='multinomial',
#        solver='lbfgs',
#    )
#    clf.fit(X_train_pca, y_train)
#
#    train_acc = clf.score(X_train_pca, y_train)
#    val_acc = clf.score(X_val_pca, y_val)
#    print(f"Train accuracy: {train_acc:.4f}")
#    print(f"Val accuracy:   {val_acc:.4f}")
#
#    return scaler, pca, clf
#
#
## --- Usage ---
#device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#scaler, pca, clf = build_pca_logreg_pipeline(train_loader, val_loader, device, n_components=10)

In [ ]:
import torch
import torch.nn as nn
from open_clip import create_model_from_pretrained


class ImageEncoder(nn.Module):
    def __init__(self, embed_dim=768):
        super().__init__()
        model, _ = create_model_from_pretrained(
            'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
        )
        self.backbone = model.visual.trunk
        biomed_dim = self.backbone.embed_dim  # 768 for ViT-B/16
        self.fc = nn.Linear(biomed_dim, embed_dim)
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x):
        features = self.backbone(x)  # (B, 768)
        return self.fc(features)


class LocaliserEncoder(nn.Module):
    def __init__(self, embed_dim=768):
        super().__init__()
        model, _ = create_model_from_pretrained(
            'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
        )
        self.backbone = model.visual.trunk
        biomed_dim = self.backbone.embed_dim
        self.fc = nn.Linear(biomed_dim, embed_dim)
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x):
        features = self.backbone(x)
        return self.fc(features)


class TabularMLP(nn.Module):
    def __init__(self, tab_dim, d_model, dropout=0.3):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(tab_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, d_model),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, tab):
        return self.mlp(tab)


class OCT_Pred(nn.Module):
    def __init__(self, n_labels, tab_dim, embed_dim=768, d_model=None, dropout=0.3):
        super().__init__()
        if d_model is None:
            d_model = embed_dim

        self.image_encoder = ImageEncoder(embed_dim)
        self.localiser_encoder = LocaliserEncoder(embed_dim)
        self.tab_encoder = TabularMLP(tab_dim, d_model, dropout)

        fusion_dim = 2 * embed_dim + d_model

        self.mlp = nn.Sequential(
            nn.Linear(fusion_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, n_labels),
        )

    def forward(self, image, localiser, tab):
        img_features = self.image_encoder(image)
        loc_features = self.localiser_encoder(localiser)
        tab_features = self.tab_encoder(tab)

        combined = torch.cat([img_features, loc_features, tab_features], dim=-1)
        output = self.mlp(combined)
        return output, combined


class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)

In [ ]:
from sklearn.metrics import classification_report
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_predicted = []

    pbar = tqdm(loader, desc="Training", leave=False)
    for images, localisers, tab, labels in pbar:
        images = images.to(device)
        localisers = localisers.to(device)
        tab = tab.to(device)
        labels = labels.long().to(device)

        optimizer.zero_grad()
        outputs, _ = model(images, localisers, tab)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_predicted.extend(predicted.cpu().numpy())

    accuracy = 100. * correct / total
    print(classification_report(all_labels, all_predicted))
    return running_loss / len(loader), accuracy

In [ ]:
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix, cohen_kappa_score, classification_report
import numpy as np

def specificity(class_ground_truth, class_prediction):
    eps = 0.000001
    cnf_matrix = confusion_matrix(class_ground_truth, class_prediction)
    FP = cnf_matrix.sum(axis=0) - np.diag(cnf_matrix)
    FN = cnf_matrix.sum(axis=1) - np.diag(cnf_matrix)
    TP = np.diag(cnf_matrix)
    TN = cnf_matrix.sum() - (FP + FN + TP)
    FP, FN, TP, TN = FP.astype(float), FN.astype(float), TP.astype(float), TN.astype(float)
    spe = TN / (TN + FP + eps)
    return spe.mean()

def evaluate(model, dataloader, criterion, device, show_report=True):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Evaluation', leave=False)
        for images, localisers, tab, labels in pbar:
            images = images.to(device)
            localisers = localisers.to(device)
            tab = tab.to(device)
            labels = labels.long().to(device)

            outputs, _ = model(images, localisers, tab)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100. * correct / total
    f1 = f1_score(all_labels, all_preds, average='micro')
    rk_corr = matthews_corrcoef(all_labels, all_preds)
    spec = specificity(all_labels, all_preds)
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    score = 0.1 * f1 + 0.1 * spec + 0.6 * qwk + 0.2 * rk_corr

    if show_report:
        print("\n" + "="*80)
        print("CLASSIFICATION REPORT - Per-Class Performance:")
        print("="*80)
        print(classification_report(all_labels, all_preds,
                                    target_names=['Class 0', 'Class 1', 'Class 2'],
                                    digits=4))
        print("="*80 + "\n")

    return {
        'loss': total_loss / len(dataloader),
        'accuracy': accuracy,
        'F1_score': f1,
        'Rk-correlation': rk_corr,
        'Specificity': spec,
        'Quadratic-weighted_Kappa': qwk,
        'score': score
    }




In [ ]:
#class FocalLoss(nn.Module):
#import torch.nn.functional as F
#    def __init__(self, gamma=2):
#        super().__init__()
#        self.gamma = gamma
#        
#    def forward(self, inputs, targets):
#        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
#        pt = torch.exp(-ce_loss)
#        focal_loss = (1 - pt) ** self.gamma * ce_loss
#        return focal_loss.mean()
#criterion = FocalLoss(gamma=2)


In [ ]:
model = OCT_Pred(n_labels=n_labels, tab_dim=7).to(device)
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
num_epochs = 5

best_score = -float('inf')

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"{'='*50}")

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.2f}%")
    print(f"Val Score: {val_metrics['score']:.4f}")

    if val_metrics['score'] > best_score:
        best_score = val_metrics['score']
        best_model=model
        print(f"New best model saved! Score: {best_score:.4f}")


In [ ]:
results = evaluate(best_model, test_loader, criterion, device)
print(f"Accuracy: {results['accuracy']:.2f}%")
print(f"F1 Score: {results['F1_score']:.4f}")
print(f"Quadratic Weighted Kappa: {results['Quadratic-weighted_Kappa']:.4f}")
print(f"Rk Correlation (Spearman): {results['Rk-correlation']:.4f}")
print(f"Specificity: {results['Specificity']:.4f}")
print(f"Score: {results['score']:.4f}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#from google.colab import files
#files.download('model.pth')
import torch
torch.save(best_model.state_dict(), '/content/drive/MyDrive/best_model_mlp.pth')